# Claude API를 사용한 Prompt caching

Prompt caching은 프롬프트의 context를 저장하고 재사용할 수 있게 하여 반복 작업의 latency를 2배 넘게 줄이고 비용을 최대 90% 줄입니다.

prompt caching을 활성화하는 방법은 두 가지입니다.

- **자동 caching**(권장): 요청의 최상위에 `cache_control` field 하나를 추가합니다. 시스템이 cache breakpoint를 자동으로 관리합니다.
- **명시적 cache breakpoint**: 무엇을 cache할지 정확하고 세밀하게 제어하려면 개별 content block에 `cache_control`을 넣습니다.

이 cookbook은 더 간단한 자동 방식부터 시작해 두 접근 방식을 모두 보여 줍니다.

## 설정

In [1]:
%pip install --upgrade 'anthropic>=0.83.0' bs4 requests python-dotenv --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import time

import anthropic
import requests
from bs4 import BeautifulSoup
from dotenv import load_dotenv

load_dotenv()
client = anthropic.Anthropic()
MODEL_NAME = "claude-sonnet-4-6"

# 이전 실행의 오래된 cache를 hit하지 않도록 고유 prefix 사용
TIMESTAMP = int(time.time())

대규모 context로 사용할 *오만과 편견* 전체 텍스트(약 187k token)를 fetch하겠습니다.

In [3]:
def fetch_article_content(url):
    response = requests.get(url, timeout=30)
    soup = BeautifulSoup(response.content, "html.parser")

    for script in soup(["script", "style"]):
        script.decompose()

    text = soup.get_text()
    lines = (line.strip() for line in text.splitlines())
    chunks = (phrase.strip() for line in lines for phrase in line.split("  "))
    text = "\n".join(chunk for chunk in chunks if chunk)

    return text


book_url = "https://www.gutenberg.org/cache/epub/1342/pg1342.txt"
book_content = fetch_article_content(book_url)

print(f"Fetched {len(book_content)} characters from the book.")
print("First 500 characters:")
print(book_content[:500])

Fetched 737526 characters from the book.
First 500 characters:
The Project Gutenberg eBook of Pride and Prejudice
This ebook is for the use of anyone anywhere in the United States and
most other parts of the world at no cost and with almost no restrictions
whatsoever. You may copy it, give it away or re-use it under the terms
of the Project Gutenberg License included with this ebook or online
at www.gutenberg.org. If you are not located in the United States,
you will have to check the laws of the country where you are located
before using this eBook.
Title:


사용량 통계를 출력할 작은 helper도 정의합니다.

In [4]:
def print_usage(response, elapsed):
    """API 응답의 token 사용량과 시간을 출력합니다."""
    usage = response.usage
    cache_create = getattr(usage, "cache_creation_input_tokens", 0)
    cache_read = getattr(usage, "cache_read_input_tokens", 0)

    print(f"  Time:                {elapsed:.2f}s")
    print(f"  Input tokens:        {usage.input_tokens}")
    print(f"  Output tokens:       {usage.output_tokens}")
    if cache_create:
        print(f"  Cache write tokens:  {cache_create}")
    if cache_read:
        print(f"  Cache read tokens:   {cache_read}")

---
## 예제 1: 자동 caching (single turn)

자동 caching은 시작하기에 가장 쉬운 방법입니다. `messages.create()` 호출의 **최상위**에 `cache_control={"type": "ephemeral"}`을 추가하면 나머지는 시스템이 처리합니다. 마지막 cache 가능한 block에 cache breakpoint를 자동으로 배치합니다.

세 가지 scenario를 비교합니다.
1. **Caching 없음** — 기준
2. **첫 cached 호출** — cache entry 생성(기준과 비슷한 시간)
3. **두 번째 cached 호출** — cache에서 읽기(큰 속도 향상)

### 기준: caching 없음

In [5]:
start = time.time()
baseline_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    messages=[
        {
            "role": "user",
            "content": str(TIMESTAMP)
            + "<book>"
            + book_content
            + "</book>"
            + "\n\nWhat is the title of this book? Only output the title.",
        }
    ],
)
baseline_time = time.time() - start

print(f"Response: {baseline_response.content[0].text}")
print_usage(baseline_response, baseline_time)

Response: Pride and Prejudice
  Time:                4.89s
  Input tokens:        187364
  Output tokens:       8


### 자동 caching을 사용하는 첫 호출 (cache write)

유일한 변경은 최상위 `cache_control` parameter입니다. 첫 호출은 cache에 쓰므로 시간은 기준과 비슷합니다.

In [6]:
start = time.time()
write_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    cache_control={"type": "ephemeral"},  # <-- 한 줄 변경
    messages=[
        {
            "role": "user",
            "content": str(TIMESTAMP)
            + "<book>"
            + book_content
            + "</book>"
            + "\n\nWhat is the title of this book? Only output the title.",
        }
    ],
)
write_time = time.time() - start

print(f"Response: {write_response.content[0].text}")
print_usage(write_response, write_time)

Response: Pride and Prejudice
  Time:                4.28s
  Input tokens:        3
  Output tokens:       8
  Cache write tokens:  187361


### 자동 caching을 사용하는 두 번째 호출 (cache hit)

같은 요청을 다시 보냅니다. 이번에는 cached prefix를 재사용하므로 상당한 속도 향상이 보일 것입니다.

In [7]:
start = time.time()
hit_response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    cache_control={"type": "ephemeral"},
    messages=[
        {
            "role": "user",
            "content": str(TIMESTAMP)
            + "<book>"
            + book_content
            + "</book>"
            + "\n\nWhat is the title of this book? Only output the title.",
        }
    ],
)
hit_time = time.time() - start

print(f"Response: {hit_response.content[0].text}")
print_usage(hit_response, hit_time)

print("\n" + "=" * 50)
print("COMPARISON")
print("=" * 50)
print(f"No caching:     {baseline_time:.2f}s")
print(f"Cache write:    {write_time:.2f}s")
print(f"Cache hit:      {hit_time:.2f}s")
print(f"Speedup:        {baseline_time / hit_time:.1f}x")

Response: Pride and Prejudice
  Time:                1.48s
  Input tokens:        3
  Output tokens:       8
  Cache read tokens:   187361

COMPARISON
No caching:     4.89s
Cache write:    4.28s
Cache hit:      1.48s
Speedup:        3.3x


---
## 예제 2: Multi-turn 대화에서 자동 caching

자동 caching은 multi-turn 대화에서 특히 빛납니다. 대화가 늘어남에 따라 cache breakpoint가 **자동으로 앞으로 이동**하므로 marker를 직접 관리할 필요가 없습니다.

| 요청 | Cache 동작 |
|---------|----------------|
| 요청 1 | System + User:A를 cache(write) |
| 요청 2 | System + User:A를 cache에서 읽음; Asst:B + User:C를 cache에 씀 |
| 요청 3 | System부터 User:C까지 cache에서 읽음; Asst:D + User:E를 cache에 씀 |

In [8]:
system_message = f"{TIMESTAMP} <file_contents> {book_content} </file_contents>"

questions = [
    "What is the title of this novel?",
    "Who are Mr. and Mrs. Bennet?",
    "What is Netherfield Park?",
    "What is the main theme of this novel?",
]

conversation = []

for i, question in enumerate(questions, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}: {question}")
    print("=" * 50)

    conversation.append({"role": "user", "content": question})

    start = time.time()
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=300,
        cache_control={"type": "ephemeral"},  # 자동 caching
        system=system_message,
        messages=conversation,
    )
    elapsed = time.time() - start

    assistant_reply = response.content[0].text
    conversation.append({"role": "assistant", "content": assistant_reply})

    print(f"\nAssistant: {assistant_reply[:200]}{'...' if len(assistant_reply) > 200 else ''}")
    print()
    print_usage(response, elapsed)


Turn 1: What is the title of this novel?

Assistant: The title of this novel is **Pride and Prejudice**, written by **Jane Austen**.

  Time:                5.19s
  Input tokens:        3
  Output tokens:       24
  Cache write tokens:  187361

Turn 2: Who are Mr. and Mrs. Bennet?

Assistant: Mr. and Mrs. Bennet are a married couple who are central characters in the novel. They live at **Longbourn** and are the parents of **five daughters**: Jane, Elizabeth, Mary, Catherine (Kitty), and Ly...

  Time:                8.27s
  Input tokens:        3
  Output tokens:       272
  Cache write tokens:  38
  Cache read tokens:   187361

Turn 3: What is Netherfield Park?

Assistant: **Netherfield Park** is a large estate located near the village of **Longbourn** in Hertfordshire, where the Bennet family lives. It plays an important role in the novel as it is the home that is let ...

  Time:                8.74s
  Input tokens:        3
  Output tokens:       300
  Cache write tokens:  283
  C

첫 turn 이후 모든 후속 turn에서 input token의 거의 100%를 cache에서 읽습니다. 대화 코드는 단순한 message 목록이며 개별 block에 특별한 `cache_control` marker가 필요하지 않습니다.

---
## 예제 3: 명시적 cache breakpoint

더 제어하려면 개별 content block에 `cache_control`을 직접 배치할 수 있습니다. 다음 경우에 유용합니다.

- 서로 다른 section을 서로 다른 TTL로 cache하고 싶음
- message content와 별개로 system prompt를 cache해야 함
- 무엇을 cache할지 세밀하게 제어하고 싶음

두 접근 방식을 결합할 수도 있습니다. system prompt에는 명시적 breakpoint를 사용하고 대화는 자동 caching이 처리하게 합니다.

아래에서는 책 content block에 `cache_control`을 직접 배치하고 매 turn마다 breakpoint를 수동으로 앞으로 이동합니다.

In [9]:
class ConversationWithExplicitCaching:
    """마지막 user message에 cache_control을 수동 배치하는 multi-turn 대화."""

    def __init__(self):
        self.turns = []

    def add_user(self, content):
        self.turns.append({"role": "user", "content": [{"type": "text", "text": content}]})

    def add_assistant(self, content):
        self.turns.append({"role": "assistant", "content": [{"type": "text", "text": content}]})

    def get_messages(self):
        """마지막 user message에 cache_control이 있는 message들을 반환합니다."""
        result = []
        last_user_idx = max(i for i, t in enumerate(self.turns) if t["role"] == "user")

        for i, turn in enumerate(self.turns):
            if i == last_user_idx:
                result.append(
                    {
                        "role": "user",
                        "content": [
                            {
                                "type": "text",
                                "text": turn["content"][0]["text"],
                                "cache_control": {"type": "ephemeral"},
                            }
                        ],
                    }
                )
            else:
                result.append(turn)

        return result


conv = ConversationWithExplicitCaching()

for i, question in enumerate(questions, 1):
    print(f"\n{'=' * 50}")
    print(f"Turn {i}: {question}")
    print("=" * 50)

    conv.add_user(question)

    start = time.time()
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=300,
        system=[
            {
                "type": "text",
                "text": system_message,
                "cache_control": {"type": "ephemeral"},  # system에 명시적 breakpoint
            },
        ],
        messages=conv.get_messages(),
    )
    elapsed = time.time() - start

    assistant_reply = response.content[0].text
    conv.add_assistant(assistant_reply)

    print(f"\nAssistant: {assistant_reply[:200]}{'...' if len(assistant_reply) > 200 else ''}")
    print()
    print_usage(response, elapsed)


Turn 1: What is the title of this novel?

Assistant: The title of this novel is **Pride and Prejudice**, written by **Jane Austen**.

  Time:                4.53s
  Input tokens:        3
  Output tokens:       24
  Cache read tokens:   187361

Turn 2: Who are Mr. and Mrs. Bennet?

Assistant: Mr. and Mrs. Bennet are a married couple who are central characters in the novel. They live at **Longbourn** and are the parents of **five daughters**: Jane, Elizabeth (Lizzy), Mary, Catherine (Kitty)...

  Time:                7.57s
  Input tokens:        3
  Output tokens:       283
  Cache read tokens:   187399

Turn 3: What is Netherfield Park?

Assistant: **Netherfield Park** is a large estate located near the village of **Longbourn** in Hertfordshire, where the Bennet family lives. It plays an important role in the novel as it is the residence that se...

  Time:                6.85s
  Input tokens:        3
  Output tokens:       300
  Cache write tokens:  294
  Cache read tokens:   187399

---
## 접근 방식 선택

| | 자동 caching | 명시적 breakpoint |
|---|---|---|
| **사용 편의성** | 한 줄 변경 | `cache_control` marker를 배치하고 이동해야 함 |
| **Multi-turn** | Breakpoint가 자동으로 앞으로 이동 | Breakpoint 배치를 직접 관리 |
| **세밀한 제어** | 불가 | 최대 4개의 독립 breakpoint |
| **혼합 TTL** | 자동 breakpoint에 단일 TTL | Breakpoint마다 서로 다른 TTL |
| **결합 가능** | 가능 — 자동 + 명시적 방식 함께 사용 | 가능 |

**자동 caching부터 시작하세요.** 최소한의 노력으로 대부분의 사용 사례를 처리합니다. 세밀한 제어가 필요할 때만 명시적 breakpoint로 전환하세요.

### 핵심 세부 사항

- **최소 cache 가능 길이:** Sonnet은 1,024 token; Opus와 Haiku 4.5는 4,096 token
- **Cache TTL:** 기본 5분(매 hit마다 갱신). 기본 input 가격의 2배로 1시간 TTL 사용 가능.
- **가격:** Cache write는 기본 input 가격의 1.25배. Cache read는 기본 input 가격의 0.1배.
- **Breakpoint 제한:** 요청당 최대 4개의 명시적 breakpoint. 자동 caching은 한 slot 사용.

전체 세부 사항은 [prompt caching 문서](https://docs.anthropic.com/en/docs/build-with-claude/prompt-caching)를 참조하세요.